In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")

In [ ]:
# Import the necessary libraries
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import (classification_report, confusion_matrix,
    average_precision_score, log_loss, precision_recall_curve)

In [ ]:
# create dir variable for data directory
DATA_DIR = Path("/Users/ashish/Downloads/claims_triage/data/raw")
OUTPUT_DIR = Path("/Users/ashish/Downloads/claims_triage/outputs/notebook_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
SAMPLE_MEMBERS = 2000
CHUNK_SIZE = 50000
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 40)
print("Read data from:", DATA_DIR.resolve())

### Read Data



In [ ]:
files = [p for p in DATA_DIR.iterdir() if p.suffix.lower() in [".csv", ".zip"]]
file_table = pd.DataFrame({"filename": [p.name for p in files],
                           "MB": [round(p.stat().st_size / 1e6, 2) for p in files]})
display(file_table)
assert len(files) >= 8, "Expected the eight Sample 1 datasets."     

b08_files = [p for p in files if "2008_beneficiary" in p.name.lower()]
b09_files = [p for p in files if "2009_beneficiary" in p.name.lower()]
b10_files = [p for p in files if "2010_beneficiary" in p.name.lower()]
ca_files = [p for p in files if "carrier" in p.name.lower() and "sample_1a." in p.name.lower()]
cb_files = [p for p in files if "carrier" in p.name.lower() and "sample_1b." in p.name.lower()]
ip_files = [p for p in files if "inpatient" in p.name.lower()]
op_files = [p for p in files if "outpatient" in p.name.lower()]
pde_files = [p for p in files if "prescription" in p.name.lower()]
for candidates in [b08_files,b09_files,b10_files,ca_files,cb_files,ip_files,op_files,pde_files]:
    assert len(candidates) == 1, f"Missing or duplicate dataset: {candidates}"
for p in b08_files+b09_files+b10_files+ip_files+op_files+pde_files:
    assert "sample_1." in p.name.lower(), f"Use Sample 1 only: {p.name}"

#### Load 2008 beneficiary data

In [ ]:
beneficiary_2008 = pd.read_csv(b08_files[0], dtype="string")
print(beneficiary_2008.shape)
display(beneficiary_2008.head())

#### Load 2009 beneficiary data

In [ ]:
beneficiary_2009 = pd.read_csv(b09_files[0], dtype="string")
print(beneficiary_2009.shape)
display(beneficiary_2009.head())

#### Load 2010 beneficiary data

In [ ]:
beneficiary_2010 = pd.read_csv(b10_files[0], dtype="string")
print(beneficiary_2009.shape)
display(beneficiary_2009.head())

In [ ]:
beneficiary_2010['DESYNPUF_ID'].value_counts() # chceck unique keys in 2010 beneficiary data

#### Select members, then keep all their histories

In [ ]:
member_ids = pd.concat([beneficiary_2008["DESYNPUF_ID"],
                        beneficiary_2009["DESYNPUF_ID"],
                        beneficiary_2010["DESYNPUF_ID"]]).dropna().drop_duplicates().sort_values()
selected_members = set(rng.choice(member_ids.to_numpy(),
    size=min(SAMPLE_MEMBERS, len(member_ids)), replace=False))
beneficiary_2008 = beneficiary_2008[beneficiary_2008.DESYNPUF_ID.isin(selected_members)].copy()
beneficiary_2009 = beneficiary_2009[beneficiary_2009.DESYNPUF_ID.isin(selected_members)].copy()
beneficiary_2010 = beneficiary_2010[beneficiary_2010.DESYNPUF_ID.isin(selected_members)].copy()
print("Sampled members:", len(selected_members))

In [ ]:
beneficiary_2008.shape, beneficiary_2009.shape, beneficiary_2010.shape

#### Carrier claims part 1

In [ ]:
pieces = []
for chunk in pd.read_csv(ca_files[0], dtype="string", chunksize=CHUNK_SIZE):
    pieces.append(chunk[chunk.DESYNPUF_ID.isin(selected_members)].copy())
carrier_1 = pd.concat(pieces, ignore_index=True)
del pieces
print(carrier_1.shape)
display(carrier_1.head())

#### Carrier claims part 2

In [ ]:
pieces = []
for chunk in pd.read_csv(cb_files[0], dtype="string", chunksize=CHUNK_SIZE):
    pieces.append(chunk[chunk.DESYNPUF_ID.isin(selected_members)].copy())
carrier_2 = pd.concat(pieces, ignore_index=True)
del pieces
print(carrier_2.shape)
display(carrier_2.head())

#### Inpatient claims

In [ ]:
pieces = []
for chunk in pd.read_csv(ip_files[0], dtype="string", chunksize=CHUNK_SIZE):
    pieces.append(chunk[chunk.DESYNPUF_ID.isin(selected_members)].copy())
inpatient = pd.concat(pieces, ignore_index=True)
del pieces
print(inpatient.shape)
display(inpatient.head())

#### Outpatient claims

In [ ]:
pieces = []
for chunk in pd.read_csv(op_files[0], dtype="string", chunksize=CHUNK_SIZE):
    pieces.append(chunk[chunk.DESYNPUF_ID.isin(selected_members)].copy())
outpatient = pd.concat(pieces, ignore_index=True)
del pieces
print(outpatient.shape)
display(outpatient.head())

#### Prescription drug events

In [ ]:
pieces = []
for chunk in pd.read_csv(pde_files[0], dtype="string", chunksize=CHUNK_SIZE):
    pieces.append(chunk[chunk.DESYNPUF_ID.isin(selected_members)].copy())
pde = pd.concat(pieces, ignore_index=True)
del pieces
print(pde.shape)
display(pde.head())

#### Combine carrier parts and remove exact ingestion duplicates

In [ ]:
carrier = pd.concat([carrier_1, carrier_2], ignore_index=True)
before = len(carrier)
carrier = carrier.drop_duplicates().copy()
print("Exact duplicate rows removed:", before-len(carrier))
assert not carrier.CLM_ID.duplicated().any(), "Conflicting carrier claim IDs; investigate raw records."
del carrier_1, carrier_2
print("Carrier claims:", len(carrier))

#### Clean beneficiary records
One beneficiary-year row per member. Only date of birth is taken as a static attribute; annual conditions enter later as prior-year summaries.

In [ ]:
beneficiary_2008["summary_year"] = 2008
beneficiary_2009["summary_year"] = 2009
beneficiary_2010["summary_year"] = 2010
beneficiaries = pd.concat([beneficiary_2008, beneficiary_2009, beneficiary_2010], ignore_index=True).drop_duplicates()
assert not beneficiaries.duplicated(["DESYNPUF_ID", "summary_year"]).any(), "Conflicting beneficiary-year rows."
beneficiaries["birth_date"] = pd.to_datetime(beneficiaries["BENE_BIRTH_DT"], format="%Y%m%d", errors="coerce")
chronic_columns = [c for c in beneficiaries.columns if c.startswith("SP_")]
beneficiaries["chronic_count"] = beneficiaries[chronic_columns].eq("1").sum(axis=1)
# A missing summary is not the same as having no chronic conditions.
beneficiaries.loc[beneficiaries[chronic_columns].isna().all(axis=1), "chronic_count"] = np.nan
static_members = beneficiaries.sort_values("summary_year").drop_duplicates("DESYNPUF_ID")[["DESYNPUF_ID", "birth_date"]]
display(beneficiaries[["DESYNPUF_ID", "summary_year", "birth_date", "chronic_count"]].head())

#### Clean carrier dates and medical codes; quarantine invalid dates

In [ ]:
for column in ["CLM_FROM_DT", "CLM_THRU_DT"]:
    carrier[column] = pd.to_datetime(carrier[column], format="%Y%m%d", errors="coerce")
invalid_dates = carrier.CLM_FROM_DT.isna() | carrier.CLM_THRU_DT.isna() | (carrier.CLM_THRU_DT < carrier.CLM_FROM_DT)
quarantine = carrier.loc[invalid_dates, ["CLM_ID", "DESYNPUF_ID", "CLM_FROM_DT", "CLM_THRU_DT"]].copy()
quarantine.to_csv(OUTPUT_DIR / "quarantined_carrier_dates.csv", index=False) # add this line to save quarantined records for later review
carrier = carrier.loc[~invalid_dates].copy()
code_columns = [c for c in carrier.columns if c.startswith(("ICD9_", "LINE_ICD9_", "HCPCS_", "PRF_PHYSN_NPI_"))]
for column in code_columns:
    carrier[column] = carrier[column].str.strip().str.upper().str.replace(".", "", regex=False).replace("", pd.NA)
print("Invalid carrier dates:", len(quarantine))

#### Convert carrier wide line slots into a line table
Suffix alignment keeps each procedure associated with its provider and diagnosis. Allowed/payment amounts are source fields used for simulation or historical records; current claim adjudication fields will not be model inputs.

In [ ]:
line_tables = []
for slot in range(1, 14):
    if f"HCPCS_CD_{slot}" not in carrier:
        continue
    line_table = carrier[["CLM_ID", "DESYNPUF_ID", "CLM_FROM_DT", "CLM_THRU_DT"]].copy()
    line_table["slot"] = slot
    line_table["procedure"] = carrier[f"HCPCS_CD_{slot}"]
    line_table["provider"] = carrier.get(f"PRF_PHYSN_NPI_{slot}", pd.Series(pd.NA, index=carrier.index)).fillna("UNKNOWN")
    line_table["line_diagnosis"] = carrier.get(f"LINE_ICD9_DGNS_CD_{slot}", pd.Series(pd.NA, index=carrier.index))
    line_table["allowed_amount"] = pd.to_numeric(carrier.get(f"LINE_ALOWD_CHRG_AMT_{slot}", pd.Series(np.nan,index=carrier.index)), errors="coerce")
    line_table["paid_amount"] = pd.to_numeric(carrier.get(f"LINE_NCH_PMT_AMT_{slot}", pd.Series(np.nan,index=carrier.index)), errors="coerce")
    line_tables.append(line_table[line_table.procedure.notna()])
lines = pd.concat(line_tables, ignore_index=True)
print(lines.shape)
display(lines.head())

#### Clean inpatient data; count each claim once despite repeated segments

In [ ]:
inpatient = inpatient.drop_duplicates().copy()
for column in ["CLM_FROM_DT", "CLM_THRU_DT"]:
    inpatient[column] = pd.to_datetime(inpatient[column], format="%Y%m%d", errors="coerce")
inpatient["CLM_PMT_AMT"] = pd.to_numeric(inpatient["CLM_PMT_AMT"], errors="coerce")
ip_bad = inpatient.CLM_FROM_DT.isna() | inpatient.CLM_THRU_DT.isna() | (inpatient.CLM_THRU_DT < inpatient.CLM_FROM_DT)
print("Invalid inpatient dates:", ip_bad.sum())
inpatient = inpatient.loc[~ip_bad].copy()
# Repeated segments must agree on claim-level fields before collapsing.
assert inpatient.groupby("CLM_ID")["DESYNPUF_ID"].nunique().le(1).all()
assert inpatient.groupby("CLM_ID")["CLM_PMT_AMT"].nunique().le(1).all()
inpatient_claims = inpatient.drop_duplicates("CLM_ID").copy()
inpatient_claims["history_year"] = inpatient_claims.CLM_THRU_DT.dt.year
print("Unique inpatient claims:", len(inpatient_claims))

#### Clean outpatient data

In [ ]:
outpatient = outpatient.drop_duplicates().copy()
for column in ["CLM_FROM_DT", "CLM_THRU_DT"]:
    outpatient[column] = pd.to_datetime(outpatient[column], format="%Y%m%d", errors="coerce")
outpatient["CLM_PMT_AMT"] = pd.to_numeric(outpatient["CLM_PMT_AMT"], errors="coerce")
op_bad = outpatient.CLM_FROM_DT.isna() | outpatient.CLM_THRU_DT.isna() | (outpatient.CLM_THRU_DT < outpatient.CLM_FROM_DT)
print("Invalid outpatient dates:", op_bad.sum())
outpatient = outpatient.loc[~op_bad].copy()
assert outpatient.groupby("CLM_ID")["DESYNPUF_ID"].nunique().le(1).all()
assert outpatient.groupby("CLM_ID")["CLM_PMT_AMT"].nunique().le(1).all()
outpatient_claims = outpatient.drop_duplicates("CLM_ID").copy()
outpatient_claims["history_year"] = outpatient_claims.CLM_THRU_DT.dt.year
print("Unique outpatient claims:", len(outpatient_claims))

#### Clean prescription events

In [ ]:
pde = pde.drop_duplicates().copy()
assert not pde.PDE_ID.duplicated().any(), "Conflicting prescription event IDs."
pde["SRVC_DT"] = pd.to_datetime(pde["SRVC_DT"], format="%Y%m%d", errors="coerce")
for column in ["TOT_RX_CST_AMT", "DAYS_SUPLY_NUM", "QTY_DSPNSD_NUM"]:
    pde[column] = pd.to_numeric(pde[column], errors="coerce")
print("Invalid drug dates:", pde.SRVC_DT.isna().sum())
pde = pde[pde.SRVC_DT.notna()].copy()
pde["history_year"] = pde.SRVC_DT.dt.year
display(pde.head())

### Exploratory Data Analysis (EDA)
Negative payments can reflect adjustments; do not automatically treat them as fraud. The amount distributions here are allowed charges, not submitted billed amounts.

In [ ]:
source_summary = pd.DataFrame({
    "rows": [len(beneficiaries), len(carrier), len(inpatient_claims), len(outpatient_claims), len(pde)],
    "members": [beneficiaries.DESYNPUF_ID.nunique(),carrier.DESYNPUF_ID.nunique(),
                inpatient_claims.DESYNPUF_ID.nunique(),outpatient_claims.DESYNPUF_ID.nunique(),pde.DESYNPUF_ID.nunique()]},
    index=["beneficiary_years","carrier","inpatient","outpatient","drug_events"])
display(source_summary)
display(lines[["allowed_amount","paid_amount"]].describe(percentiles=[.5,.9,.95,.99]).T)
display(lines[["procedure","provider","line_diagnosis","allowed_amount"]].isna().mean().rename("missing_rate"))
print("Unknown provider rate:", lines.provider.eq("UNKNOWN").mean())
print("Negative allowed charges:", lines.allowed_amount.lt(0).sum())
print("Zero allowed charges:", lines.allowed_amount.eq(0).sum())
source_summary.to_csv(OUTPUT_DIR / "source_summary.csv") # add this line to save source summary for later review

#### Visual EDA

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(13,8))
carrier.groupby(carrier.CLM_FROM_DT.dt.to_period("M")).size().plot(ax=axes[0,0],title="Carrier claims per service month")
np.log1p(lines.loc[lines.allowed_amount.ge(0),"allowed_amount"]).plot.hist(bins=50,ax=axes[0,1],title="Log allowed charge")
lines.procedure.value_counts().head(15).plot.bar(ax=axes[1,0],title="Common HCPCS codes")
carrier.groupby("DESYNPUF_ID").size().plot.hist(bins=40,ax=axes[1,1],title="Claims per sampled member")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "source_eda.png",dpi=140) # add this line to save source EDA plot for later review
plt.show()
# display(carrier.filter(regex="^ICD9_DGNS_CD_").stack().value_counts().head(15))

#### Construct one row per carrier claim
Keep the primary line as the provider/procedure representation; retain total line/provider counts. Full line-level scoring is a future improvement. Current paid/allowed amounts will not appear in X.

In [ ]:
line_summary = lines.groupby("CLM_ID").agg(
    line_count=("procedure","size"), provider_count=("provider","nunique"),
    source_allowed=("allowed_amount","sum"), source_paid=("paid_amount","sum"))
primary_line = lines.sort_values(["CLM_ID","slot"]).drop_duplicates("CLM_ID")[["CLM_ID","provider","procedure","line_diagnosis"]]
claim_columns = ["CLM_ID","DESYNPUF_ID","CLM_FROM_DT","CLM_THRU_DT"]
claims = carrier[claim_columns].merge(line_summary,on="CLM_ID",validate="one_to_one")
claims = claims.merge(primary_line,on="CLM_ID",validate="one_to_one")
diagnosis_columns = [c for c in carrier if c.startswith("ICD9_DGNS_CD_")]
carrier["diagnosis_count"] = carrier[diagnosis_columns].notna().sum(axis=1)
carrier["primary_diagnosis"] = carrier[diagnosis_columns].bfill(axis=1).iloc[:,0].fillna("UNKNOWN")
claims = claims.merge(carrier[["CLM_ID","diagnosis_count","primary_diagnosis"]],on="CLM_ID",validate="one_to_one")
claims["line_diagnosis"] = claims.line_diagnosis.fillna(claims.primary_diagnosis)
claims["service_days"] = (claims.CLM_THRU_DT-claims.CLM_FROM_DT).dt.days+1
claims["submitted_at"] = claims.CLM_THRU_DT + pd.Timedelta(days=1)
claims["claim_year"] = claims.submitted_at.dt.year
print("Claims without populated service lines excluded:", len(carrier)-len(claims))
display(claims.head())

#### Create prior-year inpatient history
Use 2008 completed services for 2009 submissions, and 2009 for 2010. This assumes prior-year records are available at the start of the following year; production would require real receipt timestamps and reporting delays.

In [ ]:
ip_history = inpatient_claims.groupby(["DESYNPUF_ID","history_year"]).agg(
    ip_claims_previous_year=("CLM_ID","nunique"), ip_paid_previous_year=("CLM_PMT_AMT","sum")).reset_index()
ip_history["claim_year"] = ip_history.history_year+1
ip_history = ip_history.drop(columns="history_year")
display(ip_history.head())

#### Create prior-year outpatient history

In [ ]:
op_history = outpatient_claims.groupby(["DESYNPUF_ID","history_year"]).agg(
    op_claims_previous_year=("CLM_ID","nunique"), op_paid_previous_year=("CLM_PMT_AMT","sum")).reset_index()
op_history["claim_year"] = op_history.history_year+1
op_history = op_history.drop(columns="history_year")
display(op_history.head())

#### Create prior-year prescription history

In [ ]:
drug_history = pde.groupby(["DESYNPUF_ID","history_year"]).agg(
    drug_events_previous_year=("PDE_ID","nunique"), drug_cost_previous_year=("TOT_RX_CST_AMT","sum"),
    drug_supply_previous_year=("DAYS_SUPLY_NUM","sum")).reset_index()
drug_history["claim_year"] = drug_history.history_year+1
drug_history = drug_history.drop(columns="history_year")
display(drug_history.head())

#### Create prior-year beneficiary summary and carrier history

In [ ]:
member_history = beneficiaries[["DESYNPUF_ID","summary_year","chronic_count"]].copy()
member_history["claim_year"] = member_history.summary_year+1
member_history = member_history.rename(columns={"chronic_count":"chronic_count_previous_year"}).drop(columns="summary_year")
carrier_history = claims.groupby(["DESYNPUF_ID","claim_year"]).agg(
    carrier_claims_previous_year=("CLM_ID","nunique"), carrier_paid_previous_year=("source_paid","sum")).reset_index()
carrier_history["claim_year"] = carrier_history.claim_year+1
display(member_history.head())

#### Join history tables one at a time, with cardinality checks
Never join raw inpatient/outpatient/PDE rows to raw carrier rows: that would multiply claims. Prior-year aggregates have one row per member/scoring year

In [ ]:
original_count = len(claims)
claims = claims.merge(static_members,on="DESYNPUF_ID",how="left",validate="many_to_one")
claims = claims.merge(ip_history,on=["DESYNPUF_ID","claim_year"],how="left",validate="many_to_one")
assert len(claims)==original_count
print("After inpatient join:", claims.shape)

In [ ]:
claims = claims.merge(op_history,on=["DESYNPUF_ID","claim_year"],how="left",validate="many_to_one")
assert len(claims)==original_count
print("After outpatient join:", claims.shape)

In [ ]:
claims = claims.merge(drug_history,on=["DESYNPUF_ID","claim_year"],how="left",validate="many_to_one")
assert len(claims)==original_count
print("After drug-event join:", claims.shape)

In [ ]:
claims = claims.merge(member_history,on=["DESYNPUF_ID","claim_year"],how="left",validate="many_to_one")
claims = claims.merge(carrier_history,on=["DESYNPUF_ID","claim_year"],how="left",validate="many_to_one")
assert len(claims)==original_count
claims["age"] = (claims.CLM_FROM_DT-claims.birth_date).dt.days/365.25
claims.loc[~claims.age.between(0,120),"age"] = np.nan
claims["previous_year_observed"] = claims.claim_year.gt(2008).astype(int)
count_history = ["ip_claims_previous_year","op_claims_previous_year","drug_events_previous_year","carrier_claims_previous_year"]
# No observed record means zero activity only where the prior year is included in the supplied data.
claims.loc[claims.previous_year_observed.eq(1),count_history] = claims.loc[claims.previous_year_observed.eq(1),count_history].fillna(0)
display(claims.head())

#### Simulate submission amounts and noisy scenarios
Carrier DE-SynPUF lacks submitted billed amounts. Use allowed-charge scales plus noise to synthesize them. Do not interpret this as real billing behavior. Scenarios are audit metadata, never predictors.

In [ ]:
positive_training = claims.loc[(claims.submitted_at < "2009-01-01") & claims.source_allowed.gt(0)]
scale_by_procedure = positive_training.groupby("procedure").source_allowed.median()
fallback_scale = positive_training.source_allowed.median()
assert pd.notna(fallback_scale), "No positive training-period amount scale."
scale = claims.source_allowed.where(claims.source_allowed.gt(0))
scale = scale.fillna(claims.procedure.map(scale_by_procedure)).fillna(fallback_scale)
claims["submitted_amount"] = scale*rng.lognormal(.25,.25,len(claims))
claims["scenario"] = rng.choice(["ordinary","inflation","pairing","burst"],size=len(claims),p=[.90,.04,.03,.03])
inflated = claims.scenario.eq("inflation")
claims.loc[inflated,"submitted_amount"] *= rng.uniform(2,7,inflated.sum())
paired = claims.scenario.eq("pairing")
training_diagnoses = claims.loc[claims.submitted_at.lt("2009-01-01"),"primary_diagnosis"].unique()
claims.loc[paired,"primary_diagnosis"] = rng.choice(training_diagnoses,paired.sum())
# Keep provider IDs out of model inputs: only their histories are encoded.
burst = claims.scenario.eq("burst")
claims.loc[burst,"provider"] = "SIM_CLUSTER_"+claims.loc[burst,"claim_year"].astype(str)
print(claims.scenario.value_counts())

#### Inject repeat submissions within the same split period
A repeated service gets a new submission ID one day later. Keep each repeat and source in the same train/calibration/validation/test calendar partition; do not copy source outcomes.

In [ ]:
claims["partition"] = np.select([
    claims.submitted_at.lt("2009-07-01"), claims.submitted_at.lt("2009-09-01"),
    claims.submitted_at.lt("2010-01-01")],["train","calibration","validation"],default="test")
repeats = claims.sample(frac=.04,random_state=SEED).copy()
repeats["CLM_ID"] = repeats.CLM_ID.astype(str)+"_REPEAT"
repeats["submitted_at"] += pd.Timedelta(days=1)
new_partition = np.select([repeats.submitted_at.lt("2009-07-01"),repeats.submitted_at.lt("2009-09-01"),
    repeats.submitted_at.lt("2010-01-01")],["train","calibration","validation"],default="test")
repeats = repeats.loc[new_partition==repeats.partition].copy()
repeats = repeats.loc[repeats.submitted_at.dt.year.eq(repeats.claim_year)].copy()
repeats["scenario"] = "duplicate"
claims = pd.concat([claims,repeats],ignore_index=True).sort_values(["submitted_at","CLM_ID"]).reset_index(drop=True)
assert claims.CLM_ID.is_unique
print("Injected repeats:", len(repeats))

#### Draw outcome targets
0 = clean, 1 = denial risk, 2 = FWA risk. Every scenario can be clean, and ordinary claims can have adverse outcomes. These are mutually exclusive simulation outcomes, not confirmed fraud. Noise reduces perfect rule recovery but does not establish independent ground truth.

In [ ]:
probability_table = pd.DataFrame([
    [.985,.012,.003], [.40,.30,.30], [.75,.23,.02], [.65,.15,.20], [.30,.65,.05]],
    index=["ordinary","inflation","pairing","burst","duplicate"],columns=["clean","denial","fwa"])
label_probabilities = probability_table.loc[claims.scenario].to_numpy()
draws = rng.random(len(claims))
claims["target"] = (draws[:,None] > np.cumsum(label_probabilities,axis=1)).sum(axis=1)
display(pd.crosstab(claims.scenario,claims.target))
display(claims.target.value_counts(normalize=True).sort_index().rename("fraction"))
claims[["CLM_ID","scenario","target","partition"]].to_csv(OUTPUT_DIR/"label_audit.csv",index=False)# add this line to save label audit for later review

#### Medical code features
ICD-9 categories preserve numeric/V/E prefixes. HCPCS buckets are broad demonstration groups, not validated clinical coverage policies. Code co-occurrence rarity does not prove diagnosis/procedure incompatibility.

In [ ]:
claims["diagnosis_group"] = claims.primary_diagnosis.astype(str).str[:3]
procedure_number = pd.Series(pd.to_numeric(claims.procedure,errors="coerce").to_numpy(dtype=float,na_value=np.nan), index=claims.index)
claims["procedure_group"] = np.select([
    procedure_number.between(99201,99499),procedure_number.between(70000,79999),
    procedure_number.between(80000,89999),procedure_number.between(90000,99199),
    procedure_number.notna()], ["evaluation_management","radiology","laboratory","medicine","other_numeric"],
    default="HCPCS_"+claims.procedure.astype(str).str[:1])
claims["log_amount"] = np.log1p(claims.submitted_amount)
claims["age_missing"] = claims.age.isna().astype(int)
claims["diagnosis_missing"] = claims.primary_diagnosis.eq("UNKNOWN").astype(int)
display(claims[["primary_diagnosis","diagnosis_group","procedure","procedure_group","log_amount"]].head())

#### Warm-up peer and code-pairing references
Freeze reference statistics before 2008-04-01 and exclude these warm-up claims from model fitting. This avoids future peer statistics influencing earlier training examples.

In [ ]:
warmup = claims.loc[claims.submitted_at.lt("2008-04-01")].copy()
assert len(warmup)>=20, "Increase SAMPLE_MEMBERS to obtain warm-up records."
peer_medians = warmup.groupby("procedure").submitted_amount.median()
global_median = warmup.submitted_amount.median()
claims["amount_peer_ratio"] = claims.submitted_amount/claims.procedure.map(peer_medians).fillna(global_median).clip(lower=1)
claims["unseen_procedure"] = (~claims.procedure.isin(peer_medians.index)).astype(int)
pair_counts = warmup.groupby(["diagnosis_group","procedure_group"]).size().rename("pair_count").reset_index()
diag_counts = warmup.groupby("diagnosis_group").size()
claims = claims.merge(pair_counts,on=["diagnosis_group","procedure_group"],how="left",validate="many_to_one")
claims["pair_rarity"] = -np.log((claims.pair_count.fillna(0)+1)/(claims.diagnosis_group.map(diag_counts).fillna(0)+10))

#### Provider daily historical counts
Rolling windows are left-closed/exclude today's submissions. Each provider is processed explicitly in a loop, without custom functions. Counts refer to the primary line provider only.

In [ ]:
claims["submission_day"] = claims.submitted_at.dt.normalize()
provider_daily = claims.groupby(["provider","submission_day"]).size().rename("daily_count").reset_index()
provider_history_parts = []
for provider_id, part in provider_daily.groupby("provider",sort=False):
    part = part.sort_values("submission_day").set_index("submission_day")
    part["provider_claims_7d"] = part.daily_count.rolling("7D",closed="left").sum().fillna(0)
    part["provider_claims_30d"] = part.daily_count.rolling("30D",closed="left").sum().fillna(0)
    provider_history_parts.append(part.reset_index())
provider_daily = pd.concat(provider_history_parts,ignore_index=True)
claims = claims.merge(provider_daily.drop(columns="daily_count"),on=["provider","submission_day"],how="left",validate="many_to_one")
claims["provider_burst_ratio"] = (claims.provider_claims_7d+1)/((claims.provider_claims_30d-claims.provider_claims_7d)/23*7+1)
display(claims[["provider","submission_day","provider_claims_7d","provider_claims_30d"]].head())

#### Member historical counts — previous 30 days, excluding today

In [ ]:
member_daily = claims.groupby(["DESYNPUF_ID","submission_day"]).size().rename("daily_count").reset_index()
member_history_parts = []
for member_id, part in member_daily.groupby("DESYNPUF_ID",sort=False):
    part = part.sort_values("submission_day").set_index("submission_day")
    part["member_claims_30d"] = part.daily_count.rolling("30D",closed="left").sum().fillna(0)
    member_history_parts.append(part.reset_index())
member_daily = pd.concat(member_history_parts,ignore_index=True)
claims = claims.merge(member_daily.drop(columns="daily_count"),on=["DESYNPUF_ID","submission_day"],how="left",validate="many_to_one")

#### Earlier matching submissions
Matching member, primary provider/procedure and service dates is a heuristic, not proof of an improper duplicate. The key uses the primary service, so additional lines/modifiers could distinguish claims.

In [ ]:
duplicate_columns = ["DESYNPUF_ID","provider","procedure","CLM_FROM_DT","CLM_THRU_DT"]
duplicate_daily = claims.groupby(duplicate_columns+["submission_day"]).size().rename("daily_count").reset_index()
duplicate_history_parts = []
for duplicate_key, part in duplicate_daily.groupby(duplicate_columns,sort=False):
    part = part.sort_values("submission_day").set_index("submission_day")
    part["previous_matches_30d"] = part.daily_count.rolling("30D",closed="left").sum().fillna(0)
    duplicate_history_parts.append(part.reset_index())
duplicate_daily = pd.concat(duplicate_history_parts,ignore_index=True)
claims = claims.merge(duplicate_daily.drop(columns="daily_count"),on=duplicate_columns+["submission_day"],how="left",validate="many_to_one")
claims = claims.sort_values(["submitted_at","CLM_ID"]).reset_index(drop=True)
print("Feature rows:", len(claims))

#### Join/feature EDA and feature allowlist
Missing earlier-year history in 2008 is intentional. Numeric imputation is fitted on training only. Do not include target, scenario, current paid/allowed amounts, IDs or annual current-year outcome summaries.

In [ ]:
numeric_features = ["log_amount","line_count","provider_count","diagnosis_count","service_days",
    "age","age_missing","diagnosis_missing","amount_peer_ratio","unseen_procedure","pair_rarity",
    "provider_claims_7d","provider_claims_30d","provider_burst_ratio","member_claims_30d","previous_matches_30d",
    "previous_year_observed","ip_claims_previous_year","ip_paid_previous_year",
    "op_claims_previous_year","op_paid_previous_year","drug_events_previous_year","drug_cost_previous_year",
    "drug_supply_previous_year","chronic_count_previous_year","carrier_claims_previous_year","carrier_paid_previous_year"]
categorical_features = ["diagnosis_group","procedure_group"]
feature_columns = numeric_features+categorical_features
assert not {"target","scenario","source_paid","source_allowed","CLM_ID","DESYNPUF_ID","provider"}.intersection(feature_columns)
display(claims[feature_columns].isna().mean().sort_values(ascending=False).rename("missing_fraction"))
display(claims.groupby("target")[["amount_peer_ratio","provider_burst_ratio","previous_matches_30d"]].median())

#### Temporal train/calibration/validation/test split
Train April 2008–June 2009, calibrate July–August 2009, select model/policy September–December 2009, test 2010. No random split and no test tuning. Training includes 2009 claims so the model can learn from available 2008 historical summaries; the previous-year-observed flag distinguishes cold-start 2008 recom

In [ ]:
model_data = claims.loc[claims.submitted_at.ge("2008-04-01") & claims.submitted_at.lt("2011-01-01")].copy()
X = model_data[feature_columns].copy()
y = model_data.target.astype(int)
train_mask = model_data.partition.eq("train")
cal_mask = model_data.partition.eq("calibration")
validation_mask = model_data.partition.eq("validation")
test_mask = model_data.partition.eq("test")
X_train,y_train = X.loc[train_mask],y.loc[train_mask]
X_cal,y_cal = X.loc[cal_mask],y.loc[cal_mask]
X_validation,y_validation = X.loc[validation_mask],y.loc[validation_mask]
X_test,y_test = X.loc[test_mask],y.loc[test_mask]
class_balance = pd.crosstab(model_data.partition,model_data.target).reindex(columns=[0,1,2],fill_value=0)
display(class_balance)
for name,target in [("train",y_train),("calibration",y_cal),("validation",y_validation),("test",y_test)]:
    assert len(target)>=50 and set(target)=={0,1,2}, f"Increase SAMPLE_MEMBERS: {name} needs all three classes."
class_balance.to_csv(OUTPUT_DIR/"class_balance.csv") # add this line to save class balance for later review
print("Train:",X_train.shape,"Calibration:",X_cal.shape,"Validation:",X_validation.shape,"Test:",X_test.shape)

#### Baseline preprocessing — scale numeric values only
Median imputation and StandardScaler are fitted inside the training pipeline. One-hot encoding caps rare code groups. Class weights address imbalance; SMOTE is avoided because synthetic interpolation of claims/codes can be incoherent.

In [ ]:
logistic_preprocessing = ColumnTransformer([
    ("numeric",Pipeline([("imputer",SimpleImputer(strategy="median",keep_empty_features=True)),
                         ("scaler",StandardScaler())]),numeric_features),
    ("codes",OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=10,max_categories=35,
                           sparse_output=False),categorical_features)])
logistic_model = Pipeline([("preprocessing",logistic_preprocessing),
    ("model",LogisticRegression(C=.5,max_iter=1500,class_weight={0:1,1:2,2:4},random_state=SEED))])
display(logistic_model)

#### Fit the baseline on training rows only

In [ ]:
logistic_model.fit(X_train,y_train)
print("Class order:",logistic_model.classes_)
print("Training accuracy (diagnostic only):",logistic_model.score(X_train,y_train))

#### Stronger model preprocessing — no scaling
Histogram gradient boosting learns nonlinear interactions through splits. Numeric scaling is unnecessary. Early stopping is disabled to avoid an internal random validation split; use a modest fixed iteration budget.

In [ ]:
boosting_preprocessing = ColumnTransformer([
    ("numeric",SimpleImputer(strategy="median",keep_empty_features=True),numeric_features),
    ("codes",OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=10,max_categories=35,
                           sparse_output=False),categorical_features)])
boosting_model = Pipeline([("preprocessing",boosting_preprocessing),
    ("model",HistGradientBoostingClassifier(max_iter=120,max_leaf_nodes=15,min_samples_leaf=25,
        learning_rate=.06,l2_regularization=2,early_stopping=False,
        class_weight={0:1,1:2,2:4},random_state=SEED))])
boosting_model.fit(X_train,y_train)
print("Boosting fitted.")

#### Calibrate both fitted models on the separate calibration period
FrozenEstimator prevents refitting the original models. Sigmoid calibration is trained on natural outcome proportions. Calibration quality still needs inspection.

In [ ]:
calibrated_logistic = CalibratedClassifierCV(FrozenEstimator(logistic_model),method="sigmoid")
calibrated_logistic.fit(X_cal,y_cal)
calibrated_boosting = CalibratedClassifierCV(FrozenEstimator(boosting_model),method="sigmoid")
calibrated_boosting.fit(X_cal,y_cal)
print("Calibrated class order:",calibrated_boosting.classes_)

#### Validation probabilities and ranking scores
Accuracy is reported later for completeness. Average precision, recall at review capacity and expected cost are more useful for rare adverse outcomes.

In [ ]:
validation_probabilities = {"logistic":calibrated_logistic.predict_proba(X_validation),
                            "boosting":calibrated_boosting.predict_proba(X_validation)}
validation_metrics = []
for model_name, probabilities in validation_probabilities.items():
    validation_metrics.append({"model":model_name,
        "adverse_AP":average_precision_score(y_validation.ne(0),probabilities[:,1]+probabilities[:,2]),
        "FWA_AP":average_precision_score(y_validation.eq(2),probabilities[:,2]),
        "log_loss":log_loss(y_validation,probabilities,labels=[0,1,2])})
display(pd.DataFrame(validation_metrics))

#### Define costs and capacity
Costs in US dollars and detection rates are explicit assumptions. Queue is a business action, not the predicted outcome class. A standard review can miss some FWA; senior review can also miss adverse outcomes. Capacities apply to sampled claims, not the complete payer population.

In [ ]:
loss_by_class = np.array([0.,200.,2000.])
review_cost = 10.
senior_cost = 30.
review_detection = np.array([0.,.90,.50])
senior_detection = np.array([0.,.95,.90])
review_capacity = 100
senior_capacity = 20
queue_names = np.array(["auto_approve","review","high_risk"])
validation_dates = model_data.loc[validation_mask,"submission_day"].to_numpy()
validation_cost_rows = []

#### Select model and routing penalty on validation only
For every day, greedily assign the largest positive avoided-loss opportunity until each review budget is exhausted. This is an approximate allocator. It is intentionally written out in this cell, not hidden in a function.

In [ ]:
for model_name, probabilities in validation_probabilities.items():
    expected_costs = np.column_stack([probabilities@loss_by_class,
        review_cost+probabilities@(loss_by_class*(1-review_detection)),
        senior_cost+probabilities@(loss_by_class*(1-senior_detection))])
    for penalty in [0,5,10,20,40,80]:
        assigned_actions = np.zeros(len(probabilities),dtype=int)
        for day in np.unique(validation_dates):
            positions = np.flatnonzero(validation_dates==day)
            benefits = expected_costs[positions,0,None]-expected_costs[positions,1:]-penalty
            order = np.argsort(-benefits.ravel(),kind="stable")
            available = {1:review_capacity,2:senior_capacity}
            for flat_index in order:
                local_position,action_index = divmod(int(flat_index),2)
                action = action_index+1
                position = positions[local_position]
                if benefits[local_position,action_index]<=0:
                    break
                if assigned_actions[position]==0 and available[action]>0:
                    assigned_actions[position]=action
                    available[action]-=1
        true_class = y_validation.to_numpy()
        detection_matrix = np.array([np.zeros(3),review_detection,senior_detection])
        actual_costs = np.array([0,review_cost,senior_cost])[assigned_actions] + loss_by_class[true_class]*(1-detection_matrix[assigned_actions,true_class])
        validation_cost_rows.append({"model":model_name,"penalty":penalty,
                                    "cost_per_1000":actual_costs.mean()*1000})
validation_cost_table = pd.DataFrame(validation_cost_rows).sort_values(["cost_per_1000","model","penalty"])
best_model_name = validation_cost_table.iloc[0]["model"]
best_penalty = float(validation_cost_table.iloc[0]["penalty"])
best_model = {"logistic":calibrated_logistic,"boosting":calibrated_boosting}[best_model_name]
display(validation_cost_table)
print("Selected:",best_model_name,"Penalty:",best_penalty)

#### Test scores — models are now frozen
Report precision/recall/F1 by outcome class as well as adverse/FWA average precision and calibration-related scores. Argmax classes here are diagnostic predictions, not routing queues.

In [ ]:
test_probabilities = {"logistic":calibrated_logistic.predict_proba(X_test),
                      "boosting":calibrated_boosting.predict_proba(X_test)}
score_rows = []
for model_name, probabilities in test_probabilities.items():
    predicted_class = probabilities.argmax(axis=1)
    print("\nMODEL:",model_name)
    print(classification_report(y_test,predicted_class,labels=[0,1,2],
        target_names=["clean","denial_risk","fwa_risk"],zero_division=0))
    display(pd.DataFrame(confusion_matrix(y_test,predicted_class,labels=[0,1,2]),
        index=["actual_clean","actual_denial","actual_fwa"],columns=["pred_clean","pred_denial","pred_fwa"]))
    score_rows.append({"model":model_name,
        "accuracy":np.mean(predicted_class==y_test.to_numpy()),
        "adverse_AP":average_precision_score(y_test.ne(0),probabilities[:,1]+probabilities[:,2]),
        "FWA_AP":average_precision_score(y_test.eq(2),probabilities[:,2]),
        "log_loss":log_loss(y_test,probabilities,labels=[0,1,2]),
        "multiclass_Brier":np.mean(np.sum((probabilities-np.eye(3)[y_test.to_numpy()])**2,axis=1))})
score_table = pd.DataFrame(score_rows)
display(score_table)
score_table.to_csv(OUTPUT_DIR/"test_scores.csv",index=False) # add this line to save test scores for later review

#### Precision–recall and reliability plots

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4))
for model_name, probabilities in test_probabilities.items():
    risk = probabilities[:,1]+probabilities[:,2]
    precision,recall,_ = precision_recall_curve(y_test.ne(0),risk)
    axes[0].plot(recall,precision,label=model_name)
    observed,predicted = calibration_curve(y_test.ne(0),risk,n_bins=8,strategy="quantile")
    axes[1].plot(predicted,observed,"o-",label=model_name)
axes[0].set(xlabel="Recall",ylabel="Precision",title="Synthetic adverse outcomes")
axes[1].plot([0,1],[0,1],"--",color="gray")
axes[1].set(xlabel="Predicted adverse probability",ylabel="Observed synthetic rate",title="Calibration")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"model_evaluation.png",dpi=140) # add this line to save model evaluation plot for later review
plt.show()

#### Route held-out test claims under daily capacity

In [ ]:
probabilities = test_probabilities[best_model_name]
test_dates = model_data.loc[test_mask,"submission_day"].to_numpy()
expected_costs = np.column_stack([probabilities@loss_by_class,
    review_cost+probabilities@(loss_by_class*(1-review_detection)),
    senior_cost+probabilities@(loss_by_class*(1-senior_detection))])
test_actions = np.zeros(len(probabilities),dtype=int)
for day in np.unique(test_dates):
    positions = np.flatnonzero(test_dates==day)
    benefits = expected_costs[positions,0,None]-expected_costs[positions,1:]-best_penalty
    order = np.argsort(-benefits.ravel(),kind="stable")
    available = {1:review_capacity,2:senior_capacity}
    for flat_index in order:
        local_position,action_index = divmod(int(flat_index),2)
        action = action_index+1
        position = positions[local_position]
        if benefits[local_position,action_index]<=0:
            break
        if test_actions[position]==0 and available[action]>0:
            test_actions[position]=action
            available[action]-=1
scored_claims = model_data.loc[test_mask,["CLM_ID","submitted_at","target"]].reset_index(drop=True)
scored_claims[["p_clean","p_denial","p_fwa"]] = probabilities
scored_claims["risk_score"] = probabilities[:,1]+probabilities[:,2]
scored_claims["queue"] = queue_names[test_actions]
display(scored_claims.head())
daily_queues = scored_claims.groupby([scored_claims.submitted_at.dt.date,"queue"]).size().unstack(fill_value=0).reindex(columns=queue_names,fill_value=0)
assert daily_queues.review.le(review_capacity).all()
assert daily_queues.high_risk.le(senior_capacity).all()
assert np.allclose(probabilities.sum(axis=1),1)
display(daily_queues.head())

#### Operational scores and cost references

In [ ]:
true_class = y_test.to_numpy()
bad = true_class!=0
reviewed = test_actions!=0
fwa = true_class==2
senior = test_actions==2
detection_matrix = np.array([np.zeros(3),review_detection,senior_detection])
routed_cost = np.array([0,review_cost,senior_cost])[test_actions]+loss_by_class[true_class]*(1-detection_matrix[test_actions,true_class])
review_all_cost = review_cost+loss_by_class[true_class]*(1-review_detection[true_class])
review_none_cost = loss_by_class[true_class]
operating_scores = pd.Series({
    "review_fraction":reviewed.mean(),
    "review_precision":bad[reviewed].mean() if reviewed.any() else 0,
    "review_recall":(bad & reviewed).sum()/max(bad.sum(),1),
    "fwa_senior_recall":(fwa & senior).sum()/max(fwa.sum(),1),
    "auto_approved_adverse_rate":bad[~reviewed].mean() if (~reviewed).any() else 0,
    "routing_cost_per_1000":routed_cost.mean()*1000,
    "review_everything_cost_per_1000":review_all_cost.mean()*1000,
    "review_nothing_cost_per_1000":review_none_cost.mean()*1000})
display(operating_scores)
operating_scores.to_csv(OUTPUT_DIR/"operating_scores.csv") # add this line to save operating scores for later review
print("Costs are conditional on synthetic labels and assumed reviewer detection, not real savings.")

#### Plain-language evidence for every reviewed/high-risk claim
These are transparent evidence alerts, not model feature attribution. They can disagree with the model; reasons do not prove fraud. The fallback states that no specific rule alert was found.

In [ ]:
reason_lists = []
for position in range(len(scored_claims)):
    reasons = []
    row = X_test.iloc[position]
    if test_actions[position]!=0:
        if row["previous_matches_30d"]>0:
            reasons.append(f"{row['previous_matches_30d']:.0f} earlier submissions match member, primary provider/procedure and service dates in 30 days; repeats can be legitimate.")
        if row["amount_peer_ratio"]>2:
            reasons.append(f"Submitted total is {row['amount_peer_ratio']:.1f} times the warm-up median for claims with this primary procedure.")
        if row["provider_burst_ratio"]>2:
            reasons.append(f"Primary provider recent submission rate is {row['provider_burst_ratio']:.1f} times its preceding-window rate.")
        if row["unseen_procedure"]==1:
            reasons.append("Primary procedure was absent from the warm-up reference; peer fallback was used.")
        if row["pair_rarity"]>3:
            reasons.append("Diagnosis/procedure groups co-occur infrequently in the warm-up data; this does not establish clinical incompatibility.")
        if not reasons:
            reasons.append("Combined model probabilities and expected loss trigger review; no specific evidence alert crossed its threshold.")
    reason_lists.append(reasons[:3])
scored_claims["reasons"] = reason_lists
display(scored_claims.loc[scored_claims.queue.ne("auto_approve"),["CLM_ID","queue","risk_score","reasons"]].head(10))

#### Save models, features, results and a direct scoring example
This bundle is a notebook model bundle, not the older API's bundle. New claims require the same historical features; do not fill them with arbitrary zeros. Below is a held-out replay using its already-built features.

In [ ]:
joblib.dump({"model":best_model,"numeric_features":numeric_features,
    "categorical_features":categorical_features,"model_name":best_model_name,"review_penalty":best_penalty,
    "peer_medians":peer_medians,"global_median":global_median,"pair_counts":pair_counts,
    "diagnosis_counts":diag_counts,"note":"Requires matching point-in-time feature preparation; not compatible with old src.api bundle."},
    OUTPUT_DIR/"notebook_model.joblib")
X.to_csv(OUTPUT_DIR/"features.csv",index=False) # add this line to save features for later review
scored_claims.to_json(OUTPUT_DIR/"test_predictions.json",orient="records",date_format="iso",indent=2)
daily_queues.to_csv(OUTPUT_DIR/"daily_queues.csv")
validation_cost_table.to_csv(OUTPUT_DIR/"validation_policy.csv",index=False)
one_claim_features = X_test.iloc[[0]].copy()
one_claim_probabilities = best_model.predict_proba(one_claim_features)[0]
display(one_claim_features)
print(dict(zip(["clean","denial_risk","fwa_risk"],one_claim_probabilities)))
print("Saved outputs:",OUTPUT_DIR.resolve()) # add this line to confirm saved outputs for later review

#### ======================= End of notebook ========================